# Lesson 2: File Handling

**Week 3 · Data Engineering Course**

---

Data engineering is mostly about moving data between files, databases, and APIs. This lesson covers the three most common file operations you will do:

1. Reading and writing CSV files
2. Reading and writing JSON
3. Navigating the file system with `pathlib`

It also covers **error handling** — what to do when a file does not exist or has a problem.

In [ ]:
import csv
import json
from pathlib import Path

RAW   = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

print('Imports ready.')

---

## 2.1 Why the `csv` Module?

You might think you can split a CSV line on commas with `line.split(',')`. For simple files that works — but real data has commas inside quoted fields, and `.split(',')` breaks on those.

In [ ]:
# A CSV row that has a comma INSIDE a quoted field
line = '1001,"Johnson, Alice",Laptop,2,1200.00'

# Naive approach: split on comma
parts = line.split(',')
print('Naive split result:')
for i, p in enumerate(parts):
    print(f'  [{i}] {p!r}')
# ['1001', '"Johnson', ' Alice"', 'Laptop', '2', '1200.00']
# The name is split into two pieces — wrong!

print()

# csv module approach: handles quoted fields correctly
import io
reader = csv.reader(io.StringIO(line))
for row in reader:
    print('csv.reader result:')
    for i, field in enumerate(row):
        print(f'  [{i}] {field!r}')
# ['1001', 'Johnson, Alice', 'Laptop', '2', '1200.00'] — correct

## 2.2 Reading a CSV File

`csv.DictReader` reads each row as a Python dictionary, with the column headers as keys.

In [ ]:
# Always use the `with` statement to open files
# It closes the file automatically when the block ends — even if an error occurs

csv_path = RAW / 'sales_q1.csv'

with open(csv_path, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f'Rows loaded: {len(rows)}')
print(f'\nFirst row:')
for key, value in rows[0].items():
    print(f'  {key}: {repr(value)}')

In [ ]:
# repr() shows the exact content — useful for spotting hidden spaces
print('customer_name (first 4 rows):')
for row in rows[:4]:
    print(f'  {repr(row["customer_name"])}')
# You may see leading/trailing spaces — those need cleaning

In [ ]:
# Walk through a sample of rows to spot data quality issues
print('Spot check — first 5 rows:')
for row in rows[:5]:
    print(f"  order {row['order_id']:>4}  "
          f"cat={row['category']!r:20}  "
          f"qty={row['quantity']!r:4}  "
          f"price={row['unit_price']!r}")

---

## 2.3 Writing a CSV File

`csv.DictWriter` writes a list of dictionaries to a CSV. You give it the column names and it writes the header and rows.

In [ ]:
out_path = CLEAN / 'sample.csv'

sample_rows = [
    {'order_id': '1001', 'product': 'Laptop',               'total': '1200.00'},
    {'order_id': '1002', 'product': 'Wireless Headphones',  'total': '150.00'},
]

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['order_id', 'product', 'total'])
    writer.writeheader()        # writes the column names as the first row
    writer.writerows(sample_rows)

print(f'Written to {out_path}')

# Read it back to verify
with open(out_path, 'r', newline='', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# If your dict has more keys than the fieldnames, use extrasaction='ignore'
# Without it, DictWriter raises a ValueError

full_row = {'order_id': '1005', 'product': 'Monitor', 'category': 'Electronics', 'total': '400.00'}

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    # We only want 3 columns in the output — ignore the extra 'category' key
    writer = csv.DictWriter(f, fieldnames=['order_id', 'product', 'total'],
                            extrasaction='ignore')
    writer.writeheader()
    writer.writerow(full_row)

print(out_path.read_text())

---

## 2.4 A Read-Clean-Write Pipeline

The basic pattern for every data cleaning script:
1. **Read** raw rows from the source file
2. **Clean** each row
3. **Write** the clean rows to a new file

In [ ]:
def clean_row(row):
    '''Clean one row. Returns the cleaned dict.'''
    row['customer_name'] = row['customer_name'].strip()
    row['product']       = row['product'].strip()
    row['category']      = row['category'].strip().title()
    return row

raw_path   = RAW / 'sales_q1.csv'
clean_path = CLEAN / 'sales_q1_clean.csv'

# Step 1: Read
with open(raw_path, 'r', newline='', encoding='utf-8') as f:
    raw_rows = list(csv.DictReader(f))
print(f'Read {len(raw_rows)} rows')

# Step 2: Clean
cleaned = [clean_row(row) for row in raw_rows]

# Step 3: Write
with open(clean_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(cleaned[0].keys()))
    writer.writeheader()
    writer.writerows(cleaned)

print(f'Wrote {len(cleaned)} rows to {clean_path}')

In [ ]:
# Scale up: process all three quarterly files in a loop
# Derive the output filename from the input filename automatically

all_rows = []

for input_path in sorted(RAW.glob('*.csv')):
    # Build the output path: sales_q1.csv -> sales_q1_clean.csv
    output_path = CLEAN / (input_path.stem + '_clean.csv')

    with open(input_path, 'r', newline='', encoding='utf-8') as f:
        file_rows = list(csv.DictReader(f))

    cleaned_file = [clean_row(row) for row in file_rows]

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(cleaned_file[0].keys()))
        writer.writeheader()
        writer.writerows(cleaned_file)

    all_rows.extend(cleaned_file)
    print(f'{input_path.name} -> {output_path.name}  ({len(cleaned_file)} rows)')

print(f'\nTotal rows across all files: {len(all_rows)}')

---

## 2.5 JSON

JSON is the standard format for APIs and configuration files. Python's `json` module converts between JSON and Python dicts/lists.

In [ ]:
# Python dict -> JSON file
config = {
    'database': 'sales_db',
    'host':     'localhost',
    'port':     5432,
    'tables':   ['orders', 'products', 'customers']
}

config_path = CLEAN / 'config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Saved:')
print(config_path.read_text())

In [ ]:
# JSON file -> Python dict
with open(config_path, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(type(loaded))           # <class 'dict'>
print(loaded['database'])     # sales_db
print(loaded['tables'][0])    # orders

In [ ]:
# json.dumps() / json.loads() work with strings instead of files
# You need this when receiving data from an API or storing JSON in a database

summary = {'file': 'sales_q1.csv', 'rows': 26, 'status': 'ok'}

# Convert to a JSON string
json_string = json.dumps(summary)
print(type(json_string))   # <class 'str'>
print(json_string)         # {"file": "sales_q1.csv", "rows": 26, "status": "ok"}

# Parse a JSON string back to a dict
parsed = json.loads(json_string)
print(type(parsed))        # <class 'dict'>
print(parsed['rows'])      # 26

---

## 2.6 pathlib — Working with File Paths

In [ ]:
# Build paths with /
q1_file = RAW / 'sales_q1.csv'

print(q1_file)             # ../data/raw/sales_q1.csv
print(q1_file.name)        # sales_q1.csv
print(q1_file.stem)        # sales_q1     (filename without extension)
print(q1_file.suffix)      # .csv
print(q1_file.parent)      # ../data/raw

In [ ]:
# Check what exists, list files
print(RAW.exists())     # True
print(RAW.is_dir())     # True

csv_files = sorted(RAW.glob('*.csv'))
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

In [ ]:
# Derive a clean output path from an input path
# This pattern avoids hard-coding output names for every file

for input_path in sorted(RAW.glob('*.csv')):
    # Replace 'raw' folder with 'clean', append _clean to the stem
    output_path = CLEAN / (input_path.stem + '_clean' + input_path.suffix)
    print(f'{input_path.name:20} -> {output_path}')

---

## 2.7 Handling Errors

Files are where things go wrong. `try/except` lets you handle failures without crashing the whole pipeline.

In [ ]:
def read_csv_safe(path):
    '''Read a CSV file. Returns an empty list if the file cannot be opened.'''
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'File not found: {path}')
        return []
    except PermissionError:
        print(f'No permission to read: {path}')
        return []

print(read_csv_safe(Path('missing.csv')))         # []
print(len(read_csv_safe(RAW / 'sales_q1.csv')))   # 26

In [ ]:
# try / except / else
# else runs only when the try block succeeded — no exception was raised
# Use it to separate 'read worked' logic from error handling

def load_and_report(path):
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'SKIP {path.name}: file not found')
        return []
    else:
        # This block only runs if open() succeeded
        print(f'OK   {path.name}: {len(rows)} rows loaded')
        return rows

for name in ['sales_q1.csv', 'missing.csv', 'sales_q2.csv']:
    load_and_report(RAW / name)

In [ ]:
# Handle a ValueError when converting a field inside a row
def parse_quantity(value):
    '''Convert a quantity string to int. Returns None on failure.'''
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return None

test_values = ['2', '3.0', 'N/A', '', '-1', '0']
for v in test_values:
    result = parse_quantity(v)
    print(f'  {v!r:8} -> {result}')

In [ ]:
# Process multiple files — skip those that fail
files_to_load = [
    RAW / 'sales_q1.csv',
    RAW / 'missing_file.csv',
    RAW / 'sales_q2.csv',
]

all_rows = []
for path in files_to_load:
    rows = read_csv_safe(path)
    if rows:
        all_rows.extend(rows)

print(f'\nTotal rows loaded: {len(all_rows)}')

In [ ]:
# Clean up files created in this lesson
for p in [out_path, clean_path, config_path]:
    if p.exists():
        p.unlink()
        print(f'Deleted {p.name}')

# Also remove the per-file clean CSVs created in section 2.4
for f in CLEAN.glob('*_clean.csv'):
    f.unlink()
    print(f'Deleted {f.name}')

---

## Key Takeaways

1. **Use `csv.DictReader`**, not `.split(',')`. Real CSV files have commas inside quoted fields that `.split()` handles incorrectly.
2. Always open files with `with open(...) as f:` and always specify `encoding='utf-8'`.
3. `csv.DictWriter` writes dicts to CSV. Pass `extrasaction='ignore'` when your dict has more keys than you want to write.
4. **Loop over files with `RAW.glob('*.csv')`** to process all quarterly files without repeating code.
5. **Derive output paths from input paths** using `input_path.stem + '_clean' + input_path.suffix` — avoids hard-coding filenames.
6. `json.dump(data, f)` writes to a file. `json.dumps(data)` gives you a string. Use `.loads()` and `.load()` to go the other way.
7. **`try/except`** catches errors. **`else`** runs only when no error occurred — use it to cleanly separate success logic from error handling.